# Tworzenie cech (feature engineering)

**Plan notatnika:**
1. Czym jest feature engineering
2. Cechy z daty i czasu
3. Cechy opóźnione i okno kroczące
4. Dyskretyzacja zmiennej ciągłej
5. Cechy z danych złożonych: listy i tekst
6. Podsumowanie

Dokumentacja: [scikit-learn.org/stable/modules/feature_extraction.html](https://scikit-learn.org/stable/modules/feature_extraction.html)

## 1. Czym jest feature engineering

Jakość modelu zależy od cech (kolumn), na których się uczy. **Feature engineering** to tworzenie nowych cech z istniejących danych tak, by wzorce były dla modelu łatwiejsze do znalezienia. Przykładowo z daty `2024-01-27` nie wynika wprost, że to sobota, a dla sprzedaży w sklepie to ważna informacja.

Nowe cechy tworzymy głównie za pomocą pandas. scikit-learn ma moduł `sklearn.feature_extraction` (głównie dla tekstu) oraz transformery takie jak `KBinsDiscretizer` i `MultiLabelBinarizer`.

In [1]:
import numpy as np
import pandas as pd
import sklearn

sklearn.__version__

'1.9.0'

## 2. Cechy z daty i czasu

Do przykładu użyjemy dziennej sprzedaży w sklepie z 10 kolejnych dni. Data jest w indeksie.

In [5]:
rng = np.random.default_rng(42)

df_sales = pd.DataFrame(
    {'sales': rng.integers(80, 200, size=10)},
    index=pd.date_range('2024-01-26', periods=10, freq='D', name='date'),
)
df_sales

,sales
date,
2024-01-26,90
2024-01-27,172
2024-01-28,158
2024-01-29,132
2024-01-30,131
2024-01-31,183
2024-02-01,90
2024-02-02,163
2024-02-03,104


Z indeksu czasowego można odczytać części daty (`.day`, `.month`, `.year`, `.dayofweek`, `.quarter`):

In [9]:
df_sales['day'] = df_sales.index.day
df_sales['month'] = df_sales.index.month
df_sales['year'] = df_sales.index.year
df_sales['dayofweek'] = df_sales.index.dayofweek        # 0 = poniedziałek, 6 = niedziela
df_sales['is_weekend'] = (df_sales['dayofweek'] >= 5).astype(int)
df_sales

,sales,day,month,year,dayofweek,is_weekend
date,,,,,,
2024-01-26,90,26,1,2024,4,0
2024-01-27,172,27,1,2024,5,1
2024-01-28,158,28,1,2024,6,1
2024-01-29,132,29,1,2024,0,0
2024-01-30,131,30,1,2024,1,0
2024-01-31,183,31,1,2024,2,0
2024-02-01,90,1,2,2024,3,0
2024-02-02,163,2,2,2024,4,0
2024-02-03,104,3,2,2024,5,1


**Cechy cykliczne.** Dzień tygodnia `6` (niedziela) i `0` (poniedziałek) sąsiadują ze sobą, a w danych liczbowych dzieli je odległość 6. Cykliczność oddają funkcje sinus i cosinus:

In [10]:
df_sales['dow_sin'] = np.sin(2 * np.pi * df_sales['dayofweek'] / 7)
df_sales['dow_cos'] = np.cos(2 * np.pi * df_sales['dayofweek'] / 7)
df_sales[['dayofweek', 'dow_sin', 'dow_cos']].round(2)

,dayofweek,dow_sin,dow_cos
date,,,
2024-01-26,4,-0.43,-0.90
2024-01-27,5,-0.97,-0.22
2024-01-28,6,-0.78,0.62
2024-01-29,0,0.00,1.00
2024-01-30,1,0.78,0.62
2024-01-31,2,0.97,-0.22
2024-02-01,3,0.43,-0.90
2024-02-02,4,-0.43,-0.90
2024-02-03,5,-0.97,-0.22


## 3. Cechy opóźnione i okno kroczące

W szeregach czasowych bardzo przydatne są wartości z przeszłości:
- `shift(1)` – wartość z poprzedniego dnia (*lag*),
- `rolling(3).mean()` – średnia krocząca z 3 ostatnich dni.

**Uwaga:** średnia z okna zawierającego bieżący dzień „zna” wynik, który mamy przewidzieć. Dlatego najpierw przesuwamy szereg (`shift(1)`), a potem liczymy średnią.

In [11]:
df_lag = df_sales[['sales']].copy()
df_lag['sales_lag1'] = df_lag['sales'].shift(1)
df_lag['sales_mean3'] = df_lag['sales'].shift(1).rolling(3).mean()
df_lag

,sales,sales_lag1,sales_mean3
date,,,
2024-01-26,90,NaN,NaN
2024-01-27,172,90.0,NaN
2024-01-28,158,172.0,NaN
2024-01-29,132,158.0,140.000000
2024-01-30,131,132.0,154.000000
2024-01-31,183,131.0,140.333333
2024-02-01,90,183.0,148.666667
2024-02-02,163,90.0,134.666667
2024-02-03,104,163.0,145.333333


Na początku szeregu pojawiają się braki (`NaN`), bo nie ma jeszcze historii. Obsługę braków omówiliśmy w notatniku 02 (`dropna()` lub imputacja).

## 4. Dyskretyzacja zmiennej ciągłej

**Dyskretyzacja** (*binning*) zamienia liczby na przedziały, np. wzrost na kategorie `small`, `medium`, `high`. Pomaga, gdy zależność nie jest liniowa lub gdy chcemy ograniczyć wpływ wartości odstających. Kosztem jest utrata szczegółów.

In [12]:
df_height = pd.DataFrame({'height': [175.0, 178.5, 185.0, 191.0, 184.5, 183.0, 168.0]})
df_height

,height
0,175.0
1,178.5
2,185.0
3,191.0
4,184.5
5,183.0
6,168.0


`pd.cut()` z liczbą przedziałów dzieli zakres na części o **równej szerokości**:

In [13]:
pd.cut(df_height['height'], bins=3)

0    (167.977, 175.667]
1    (175.667, 183.333]
2      (183.333, 191.0]
3      (183.333, 191.0]
4      (183.333, 191.0]
5    (175.667, 183.333]
6    (167.977, 175.667]
Name: height, dtype: category
Categories (3, interval[float64, right]): [(167.977, 175.667] < (175.667, 183.333] < (183.333, 191.0]]

Granice można podać ręcznie, wraz z nazwami przedziałów. Przedziały są domknięte z prawej strony: `(160, 175]` zawiera 175.

In [14]:
df_height['height_cat'] = pd.cut(
    df_height['height'],
    bins=[160, 175, 180, 195],
    labels=['small', 'medium', 'high'],
)
df_height

,height,height_cat
0,175.0,small
1,178.5,medium
2,185.0,high
3,191.0,high
4,184.5,high
5,183.0,high
6,168.0,small


`pd.qcut()` dzieli na przedziały o **równej liczbie obserwacji** (kwantyle):

In [15]:
pd.qcut(df_height['height'], q=3, labels=['small', 'medium', 'high'])

0     small
1     small
2      high
3      high
4    medium
5    medium
6     small
Name: height, dtype: category
Categories (3, str): ['small' < 'medium' < 'high']

Odpowiednik w scikit-learn to `KBinsDiscretizer`, który można dołączyć do `ColumnTransformer`:

In [16]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
kbins.fit_transform(df_height[['height']])

array([[0.],
       [1.],
       [2.],
       [2.],
       [2.],
       [1.],
       [0.]])

Otrzymane kategorie możemy zakodować one-hot (patrz notatnik 01):

In [17]:
pd.get_dummies(df_height, columns=['height_cat'], prefix='height', dtype=int)

,height,height_small,height_medium,height_high
0,175.0,1,0,0
1,178.5,0,1,0
2,185.0,0,0,1
3,191.0,0,0,1
4,184.5,0,0,1
5,183.0,0,0,1
6,168.0,1,0,0


## 5. Cechy z danych złożonych: listy i tekst

### Listy

Kolumna może zawierać listy (np. znane języki).

In [18]:
df_lang = pd.DataFrame({'lang': [['PL', 'ENG'], ['GER', 'ENG', 'PL', 'FRA'], ['RUS']]})
df_lang

,lang
0,"[PL, ENG]"
1,"[GER, ENG, PL, FRA]"
2,[RUS]


In [21]:
df_lang['lang'].apply(lambda x: 1 if 'PL' in x else 0) 

0    1
1    1
2    0
Name: lang, dtype: int64

In [ ]:
df_lang['lang_number'] = df_lang['lang'].apply(len)                           # liczba języków
df_lang['PL_flag'] = df_lang['lang'].apply(lambda x: 1 if 'PL' in x else 0)   # czy zna polski
df_lang

Gdy chcemy mieć kolumnę dla **każdego** języka, używamy `MultiLabelBinarizer` (odpowiednik one-hot dla list):

In [22]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
pd.DataFrame(mlb.fit_transform(df_lang['lang']), columns=mlb.classes_, index=df_lang.index)

,ENG,FRA,GER,PL,RUS
0,1,0,0,1,0
1,1,1,1,1,0
2,0,0,0,0,1


### Tekst w kolumnie – `str.split()`

In [23]:
df_web = pd.DataFrame({'website': ['wp.pl', 'onet.pl', 'google.com', 'bbc.co.uk']})
df_web

,website
0,wp.pl
1,onet.pl
2,google.com
3,bbc.co.uk


`n=1` oznacza podział tylko na pierwszej kropce. Dzięki temu `co.uk` pozostaje w jednej kolumnie.

In [24]:
df_web[['portal', 'extension']] = df_web['website'].str.split('.', n=1, expand=True)
df_web

,website,portal,extension
0,wp.pl,wp,pl
1,onet.pl,onet,pl
2,google.com,google,com
3,bbc.co.uk,bbc,co.uk


### Tekst jako liczby – `CountVectorizer`

Model nie rozumie zdań, więc trzeba je zamienić na liczby. `CountVectorizer` buduje słownik wszystkich słów i dla każdego dokumentu **liczy wystąpienia** każdego słowa (*bag of words*).

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ['Python jest świetny', 'Python i uczenie maszynowe', 'uczenie jest świetne']

vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(docs)

pd.DataFrame(counts.toarray(), columns=vectorizer.get_feature_names_out())

,jest,maszynowe,python,uczenie,świetne,świetny
0,1,0,1,0,0,1
1,0,1,1,1,0,0
2,1,0,0,1,1,0


Wariant `TfidfVectorizer` zmniejsza wagę słów występujących w wielu dokumentach (mało informacyjnych). Zaawansowane metody pracy z tekstem (m.in. modele językowe) poznamy w rozdziale 5.

## 6. Podsumowanie

| Rodzaj danych | Technika | Narzędzie |
|---|---|---|
| Data i czas | dzień, miesiąc, dzień tygodnia, weekend, sin/cos | `.index.day`, `.dayofweek`, `np.sin` |
| Szeregi czasowe | wartości opóźnione, średnia krocząca | `shift()`, `rolling()` |
| Liczby ciągłe | przedziały | `pd.cut`, `pd.qcut`, `KBinsDiscretizer` |
| Listy kategorii | flagi, liczność | `apply`, `MultiLabelBinarizer` |
| Tekst w kolumnie | podział na części | `str.split()` |
| Dłuższy tekst | liczenie słów | `CountVectorizer`, `TfidfVectorizer` |

**Najważniejsze zasady:**
1. Twórz cechy, które mają sens dla problemu (np. weekend dla sprzedaży).
2. W szeregach czasowych nie używaj informacji z przyszłości (`shift` przed `rolling`).
3. Nowe cechy mogą dawać braki i wymagają dalszego przygotowania (notatniki 01 i 02).